# **Pre-Trained Model 1 - Efficient Net B0**

EfficientNetB0 was selected as the first pretrained architecture based on three
key criteria: parameter efficiency, proven medical imaging performance and
architectural design alignment with our task.


### Imports

In [ ]:
import os
import sys
import keras
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.preprocessing import label_binarize

if os.getcwd().endswith('models'):
    os.chdir('..')
    
from utils.utils_model import *
from utils.utils_augmentation import *
from utils.utils_preproc import *

In [2]:
%pip install "numpy<2"

  Using cached numpy-1.26.4-cp310-cp310-win_amd64.whl (15.8 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [WinError 5] Acesso negado: 'C:\\Users\\faust\\AppData\\Local\\Programs\\Python\\Python310\\Lib\\site-packages\\~~mpy.libs\\libscipy_openblas64_-13e2df515630b4a41f92893938845698.dll'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### **Data** Configuration

In [3]:
# path for the new images
base_path = "./data" 
aug_dir = os.path.join(base_path, "HAM10000_augmented")
if not os.path.exists(aug_dir):
    os.makedirs(aug_dir)

In [4]:
# CORRIGIR COM BASE NO AUGMENTED

train_df = pd.read_csv('data/augmented_metadata.csv')
val_df   = pd.read_csv('data/val_split.csv')
test_df  = pd.read_csv('data/test_split.csv')

train_df = train_df.rename(columns={'cleaned_path': 'image_path'})
val_df = val_df.rename(columns={'cleaned_path': 'image_path'})
test_df = test_df.rename(columns={'cleaned_path': 'image_path'})

train_ds = make_dataset(train_df, resize_function=format_area_matched, shuffle=True, repeat=True)  
val_ds   = make_dataset(val_df, resize_function=format_area_matched)                               
test_ds  = make_dataset(test_df, resize_function=format_area_matched)

train_df['dataset'] = 'original'
val_df['dataset'] = 'original'
test_df['dataset'] = 'original'

# Add encoded labels
with open("label2idx.json", "r") as f:
    label2idx = json.load(f)

for df in [train_df, val_df, test_df]:
    df['dx_encoded'] = df['dx'].map(label2idx)

BATCH_SIZE = 32 # VERIFICAR PARA QUE É QUE ISTO SERVE
N_CLASSES  = len(label2idx) # VERIFICAR PARA QUE É QUE ISTO SERVE

NameError: name 'format_area_matched' is not defined

### **Model** Configuration

In [ ]:
UNFREEZE_FROM_B = -30   # unfreeze last 30 layers of EfficientNetB0

def build_efficientnet():
    base = keras.applications.EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_shape=(224, 224, 3),
        pooling=None
        
    )
    base.trainable = False

    inputs = keras.Input(shape=(224, 224, 3))
    x = base(inputs, training=False)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(0.4)(x)
    x = keras.layers.Dense(256, activation="relu")(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Dropout(0.3)(x)
    outputs = keras.layers.Dense(N_CLASSES)(x)
    return keras.Model(inputs, outputs, name="efficientnet_b0")

In [ ]:
# phase 1: head only
model_pt1 = build_efficientnet() # Model Pre-Trained 1
model_pt1.summary()



model_pt1.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

history1_pt1 = model_pt1.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    steps_per_epoch=STEPS_PER_EPOCH,
    class_weight=class_weights_dict,
    callbacks=get_callbacks("checkpoints/model_b_phase1.weights.h5",
                            patience_es=6, patience_lr=3)
)

plot_history(history1_pt1, "Model B — EfficientNetB0 Phase 1 (head only)")

In [ ]:
# Phase 2: Fine-tune top layers
base_b = model_pt1.layers[1]
base_b.trainable = True
for layer in base_b.layers[:UNFREEZE_FROM_B]:
    layer.trainable = False

model_pt1.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

history2_b0 = model_pt1.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    steps_per_epoch=STEPS_PER_EPOCH,
    class_weight=class_weights_dict,
    callbacks=get_callbacks("checkpoints/model_b_best.weights.h5",
                            patience_es=8, patience_lr=4)
)

plot_history(history2_pt1, "Model B — EfficientNetB0 Phase 2 (fine-tune)")
results_b = evaluate_model(model_pt1, test_ds, test_df, label2idx, "Model B — EfficientNetB0")
